## Imports

In [1]:
import wandb
import logging
from tqdm import tqdm
from wandb.sdk.wandb_run import Run
import numpy as np
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objs as go
import seaborn as sns
import matplotlib.pyplot as plt
from nn_core.common import PROJECT_ROOT
import json

/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


## Configuration

In [2]:
from mass.utils.plots import Palette

plt.rcParams.update(
    {
        "text.usetex": True,
        "font.family": "serif",
        "axes.titlesize": 24,        # Larger axes/title fonts
        "axes.labelsize": 24,
        "xtick.labelsize": 24,
        "ytick.labelsize": 20,
        "legend.fontsize": 24,
    }
)
sns.set_context("talk")

cmap_name = "coolwarm_r"

palette = Palette(f"{PROJECT_ROOT}/misc/palette.json", map_path=f"{PROJECT_ROOT}/misc/palette_map.json")
palette

Project not installed in the current env, activate the correct env or install it with:
	pip install -e .


{'blue': '#335c67',
 'white': '#fff3b0',
 'yellow': '#e09f3e',
 'red': '#9e2a2b',
 'dark red': '#540b0e',
 'green': '#81b29a'}

## Get runs

In [3]:
api = wandb.Api()
entity, project = "gladia", "mass"  # set to your entity and project

In [4]:
def get_runs(entity, project, positive_tags, negative_tags):
    filters_pos_tags = {"$and": [{"tags": {"$eq": pos_tag}} for pos_tag in positive_tags]}
    filters_neg_tags = {}

    print(filters_pos_tags)
    filters = {**filters_pos_tags, **filters_neg_tags}
    runs = api.runs(entity + "/" + project, filters=filters)

    print(f"There are {len(runs)} runs respecting these conditions.")
    return runs

In [5]:
tags = [
    "finetune",
    "rebuttal"
]  

In [6]:
runs = get_runs(entity, project, positive_tags=tags, negative_tags=[])

{'$and': [{'tags': {'$eq': 'finetune'}}, {'tags': {'$eq': 'rebuttal'}}]}
There are 13 runs respecting these conditions.


In [7]:
print(set(runs[0].history().columns))

{'_runtime', 'lr-AdamW', '_timestamp', 'loss/val/Weather', 'trainer/global_step', 'epoch', 'loss/train/Weather', 'acc/train/Weather', 'acc/test/Weather', 'acc/val/Weather', '_step', 'loss/test/Weather'}


In [15]:
models = ['ViT-B-32', 'ViT-B-16', 'ViT-L-14'] # ['ViT-B-32', 'ViT-B-16', 'ViT-L-14']
# datasets =  ['Cars', 'DTD', 'EuroSAT', 'GTSRB', 'MNIST', 'RESISC45', 'SUN397', 'SVHN', 'CIFAR100', 'STL10', 'Flowers102', 'OxfordIIITPet', 'PCAM', 'FER2013', 'EMNIST', 'CIFAR10', 'Food101', 'FashionMNIST', 'RenderedSST2', 'KMNIST']
datasets = ['Beans', 'CUB200', 'Dogs', 'FlowersKaggle', 'Fruits360', 'Garbage', 'IntelImages', 'KenyanFood13', 'KvasirV2', 'Landscape', 'MangoLeafBD', 'Vegetables', 'Weather']

In [16]:
accs = {model: {dataset: None for dataset in datasets} for model in models}

#### Hparams

In [22]:
run_by_model = {}


for run in runs:
    for model in models:
        
        # runs are tagged with 'B32', 'B16', 'L14'
        if not model.replace('-', '')[-3:] in run.tags:
            continue 
            
        cols = run.history().columns
        print(cols)
        # Quirk: config wasn't uploaded when finetuned 
        for dataset in datasets:
            for col in cols:
                col_dataset = col.split('/')[-1]
                if dataset == col_dataset:
                    if dataset == 'Fruits360':
                        print('LMAO')
                    print(f'Found dataset {dataset}')
                    accs[model][dataset] = run.history()[f'acc/test/{dataset}'].values[-1]
                    break # Found the dataset, break

Index(['lr-AdamW', '_timestamp', 'acc/val/Weather', 'acc/test/Weather',
       'acc/train/Weather', '_step', '_runtime', 'loss/test/Weather',
       'trainer/global_step', 'epoch', 'loss/val/Weather',
       'loss/train/Weather'],
      dtype='object')
Found dataset Weather
Index(['acc/val/Vegetables', 'acc/test/Vegetables', 'trainer/global_step',
       '_timestamp', 'loss/val/Vegetables', 'acc/train/Vegetables', '_runtime',
       'lr-AdamW', 'loss/train/Vegetables', '_step', 'loss/test/Vegetables',
       'epoch'],
      dtype='object')
Found dataset Vegetables
Index(['acc/val/MangoLeafBD', 'loss/val/MangoLeafBD', 'acc/train/MangoLeafBD',
       '_step', 'acc/test/MangoLeafBD', 'loss/train/MangoLeafBD', 'lr-AdamW',
       'trainer/global_step', 'loss/test/MangoLeafBD', 'epoch', '_runtime',
       '_timestamp'],
      dtype='object')
Found dataset MangoLeafBD
Index(['acc/val/Landscape', 'loss/train/Landscape', 'epoch',
       'loss/test/Landscape', '_runtime', 'lr-AdamW', 'loss/val/L

KeyboardInterrupt: 

In [21]:
accs

{'ViT-B-32': {'Beans': 0.9296875,
  'CUB200': 0.7224420309066772,
  'Dogs': 0.7583414316177368,
  'FlowersKaggle': 0.9614197611808777,
  'Fruits360': None,
  'Garbage': 0.9631578922271729,
  'IntelImages': None,
  'KenyanFood13': 0.8269938826560974,
  'KvasirV2': 0.9158333539962769,
  'Landscape': None,
  'MangoLeafBD': 1.0,
  'Vegetables': 1.0,
  'Weather': 0.9360465407371521},
 'ViT-B-16': {'Beans': None,
  'CUB200': None,
  'Dogs': None,
  'FlowersKaggle': None,
  'Fruits360': None,
  'Garbage': None,
  'IntelImages': None,
  'KenyanFood13': None,
  'KvasirV2': None,
  'Landscape': None,
  'MangoLeafBD': None,
  'Vegetables': None,
  'Weather': None},
 'ViT-L-14': {'Beans': None,
  'CUB200': None,
  'Dogs': None,
  'FlowersKaggle': None,
  'Fruits360': None,
  'Garbage': None,
  'IntelImages': None,
  'KenyanFood13': None,
  'KvasirV2': None,
  'Landscape': None,
  'MangoLeafBD': None,
  'Vegetables': None,
  'Weather': None}}

In [12]:
bench_N8 = ["SUN397", "Cars", "RESISC45", "EuroSAT",  "SVHN", "GTSRB", "MNIST", "DTD"]
bench_N14 = ["SUN397", "Cars", "RESISC45", "EuroSAT",  "SVHN", "GTSRB", "MNIST", "DTD", "Flowers102", "PCAM", "FER2013", "OxfordIIITPet", "STL10", "CIFAR100"]
bench_N20 = ["SUN397", "Cars", "RESISC45", "EuroSAT",  "SVHN", "GTSRB", "MNIST", "DTD", "Flowers102", "PCAM", "FER2013", "OxfordIIITPet", "STL10", "CIFAR100", "CIFAR10", "Food101", "FashionMNIST", "RenderedSST2", "EMNIST", "KMNIST"]

In [13]:
accs_N8 = {model: {dataset: accs[model][dataset] for dataset in bench_N8} for model in models}
accs_N14 = {model: {dataset: accs[model][dataset] for dataset in bench_N14} for model in models}
accs_N20 = {model: {dataset: accs[model][dataset] for dataset in bench_N20} for model in models}
accs_N8

{'ViT-B-32': {'SUN397': None,
  'Cars': None,
  'RESISC45': None,
  'EuroSAT': None,
  'SVHN': None,
  'GTSRB': None,
  'MNIST': None,
  'DTD': None},
 'ViT-B-16': {'SUN397': None,
  'Cars': None,
  'RESISC45': None,
  'EuroSAT': None,
  'SVHN': None,
  'GTSRB': None,
  'MNIST': None,
  'DTD': None},
 'ViT-L-14': {'SUN397': None,
  'Cars': None,
  'RESISC45': None,
  'EuroSAT': None,
  'SVHN': None,
  'GTSRB': None,
  'MNIST': None,
  'DTD': None}}

In [14]:
# take the mean 
mean_accs_N8 = {model: np.mean([accs_N8[model][dataset] for dataset in bench_N8]) for model in models}
mean_accs_N14 = {model: np.mean([accs_N14[model][dataset] for dataset in bench_N14]) for model in models}
mean_accs_N20 = {model: np.mean([accs_N20[model][dataset] for dataset in bench_N20]) for model in models}


TypeError: unsupported operand type(s) for +: 'NoneType' and 'NoneType'

In [ ]:
mean_accs_N20

{'ViT-B-32': 0.8951470673084259,
 'ViT-B-16': 0.9191412061452866,
 'ViT-L-14': 0.9400492221117019}